In [3]:
import pandas as pd
import numpy as np
import random
import time
import math

# ========= Helpers: POZ parsing & feasibility ==========

def parse_poz_cell(cell):
    """
    Parse string seperti "[210 240], [350 380]" -> list of (210.0, 240.0), (350.0, 380.0)
    Jika kosong/NaN -> []
    """
    if isinstance(cell, str):
        pairs = []
        # ambil semua sub-string di dalam [..]
        for part in cell.replace(',', ' ').split(']'):
            if '[' in part:
                nums = part.split('[')[-1].strip().split()
                if len(nums) == 2:
                    try:
                        lo, hi = float(nums[0]), float(nums[1])
                        if lo > hi:
                            lo, hi = hi, lo
                        pairs.append((lo, hi))
                    except:
                        pass
        return pairs
    return []

def is_in_poz(val, poz_list):
    """True jika val berada di dalam salah satu interval terlarang."""
    for lo, hi in poz_list:
        if lo <= val <= hi:
            return True
    return False

def repair_out_of_poz(val, lb, ub, poz_list):
    """
    Jika val berada di interval terlarang, geser ke batas terdekat dari interval tersebut.
    Pastikan tetap di [lb, ub].
    """
    if not poz_list:
        return min(max(val, lb), ub)
    v = float(min(max(val, lb), ub))
    changed = True
    # perulangan singkat: kalau setelah digeser masih masuk interval lain
    for _ in range(5):
        moved = False
        for lo, hi in poz_list:
            if lo <= v <= hi:
                # pilih sisi terdekat
                to_lo = abs(v - lo)
                to_hi = abs(hi - v)
                v = lo - 1e-9 if to_lo <= to_hi else hi + 1e-9
                v = float(min(max(v, lb), ub))
                moved = True
        if not moved:
            break
    return v

def margin_up(val, ub, poz_list):
    """
    Headroom kenaikan tanpa memasuki POZ: jarak ke batas terdekat ke atas.
    """
    m = ub - val
    for lo, hi in poz_list:
        if val < lo:
            m = min(m, lo - val)
        elif lo <= val <= hi:
            m = min(m, max(0.0, hi - val))  # akan diperbaiki di tempat lain
        # jika val > hi: tidak mempengaruhi margin langsung
    return max(0.0, m)

def margin_down(val, lb, poz_list):
    """Headroom penurunan tanpa memasuki POZ."""
    m = val - lb
    for lo, hi in poz_list:
        if val > hi:
            m = min(m, val - hi)
        elif lo <= val <= hi:
            m = min(m, max(0.0, val - lo))
    return max(0.0, m)

def redistribute_to_meet_demand(P, demand, lb, ub, poz_lists, tol=1e-6):
    """
    Greedy ringan untuk menyesuaikan sum(P) -> demand tanpa masuk POZ.
    Menyebarkan selisih ke unit-unit berdasar margin naik/turun.
    """
    P = P.copy()
    diff = demand - np.sum(P)
    if abs(diff) <= tol:
        return P

    if diff > 0:  # butuh menaikkan
        # kapasitas kenaikan aman
        ups = np.array([margin_up(P[i], ub[i], poz_lists[i]) for i in range(len(P))])
        total_up = np.sum(ups)
        if total_up <= tol:
            return P  # tidak bisa, biar penalti objektif bekerja
        frac = np.minimum(1.0, np.maximum(0.0, ups / (total_up + 1e-12)))
        step = np.minimum(ups, diff * frac + 1e-12)
        P += step
    else:        # butuh menurunkan
        downs = np.array([margin_down(P[i], lb[i], poz_lists[i]) for i in range(len(P))])
        total_dn = np.sum(downs)
        if total_dn <= tol:
            return P
        frac = np.minimum(1.0, np.maximum(0.0, downs / (total_dn + 1e-12)))
        step = np.minimum(downs, (-diff) * frac + 1e-12)
        P -= step

    # rapikan dari POZ setelah redistribusi
    for i in range(len(P)):
        P[i] = repair_out_of_poz(P[i], lb[i], ub[i], poz_lists[i])
    return P

# ========= Load data ==========

data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/6-unit.xlsx',
    sheet_name='Table 3'
)

a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values.astype(float)
Maximum_Capacity = data['pmax'].values.astype(float)
ramp_up = data['rampup'].values.astype(float)
ramp_down = data['rampdown'].values.astype(float)
Unit = data['Unit'].values

# kolom opsional Pnow & poz
Pnow = data['Pnow'].values.astype(float) if 'Pnow' in data.columns else None
poz_lists = [parse_poz_cell(cell) for cell in (data['poz'].values if 'poz' in data.columns else [np.nan]*len(Unit))]

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd6u.xlsx')
power_demand = power_demand_data['load'].values

# ========= DOA dengan POZ ==========

def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    poz_lists,
                    pop=30, max_iter=100,
                    penalty_mismatch=1e4,
                    penalty_poz=1e6):
    dim = len(pmin)
    lb, ub = pmin, pmax

    def fobj(pos):
        # biaya kuadratik biasa
        fuel = np.sum(a + b * pos + c * pos**2)
        # penalti mismatch demand
        penalty = abs(demand - np.sum(pos)) * penalty_mismatch
        # penalti POZ (jarak minimum ke tepi interval jika berada di dalam)
        poz_pen = 0.0
        for i in range(dim):
            for lo, hi in poz_lists[i]:
                if lo <= pos[i] <= hi:
                    poz_pen += penalty_poz * min(pos[i]-lo, hi-pos[i] + 1e-12)
        return fuel + penalty + poz_pen

    # inisialisasi populasi (hindari POZ)
    x = np.random.uniform(lb, ub, (pop, dim))
    for p in range(pop):
        for i in range(dim):
            x[p, i] = repair_out_of_poz(x[p, i], lb[i], ub[i], poz_lists[i])

    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration 90% ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # batas + perbaikan POZ
                        x[j, h] = float(min(max(x[j, h], lb[h]), ub[h]))
                        x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]
                        x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])

            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation 10% ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                x[j, h] = float(min(max(x[j, h], lb[h]), ub[h]))
                x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])

        fbest_history[it] = fbest

    # ====== Final scaling & repairs ======
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    # clip + keluar dari POZ
    for i in range(dim):
        sbest[i] = float(min(max(sbest[i], lb[i]), ub[i]))
        sbest[i] = repair_out_of_poz(sbest[i], lb[i], ub[i], poz_lists[i])

    # Coba sesuaikan total agar match demand tanpa masuk POZ
    sbest = redistribute_to_meet_demand(sbest, demand, lb, ub, poz_lists)

    return sbest, max_iter

# ========= Fuel cost (tanpa VPE, tetap kuadratik) =========
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# ========= Main loop =========
all_outputs = []
schedulling = []
summary_rows = []

baseline_prev = Pnow if Pnow is not None else None

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    # jalankan optimisasi
    P, iterations = calculate_power(
        demand, a, b, c,
        Minimum_Capacity, Maximum_Capacity,
        ramp_up, ramp_down,
        poz_lists=poz_lists
    )

    # Terapkan ramp-rate relatif ke baseline:
    # - untuk titik pertama: dari Pnow (jika ada), kalau tidak dari P hasil (tidak diubah)
    # - selanjutnya: dari output sebelumnya
    if baseline_prev is not None:
        diff = P - baseline_prev
        up_mask, down_mask = diff > 0, diff < 0
        P[up_mask] = baseline_prev[up_mask] + np.minimum(diff[up_mask], ramp_up[up_mask])
        P[down_mask] = baseline_prev[down_mask] + np.maximum(diff[down_mask], -ramp_down[down_mask])
        # repair POZ setelah ramp
        for i in range(len(P)):
            P[i] = repair_out_of_poz(P[i], Minimum_Capacity[i], Maximum_Capacity[i], poz_lists[i])
        # coba paskan demand lagi
        P = redistribute_to_meet_demand(P, demand, Minimum_Capacity, Maximum_Capacity, poz_lists)

    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    info = {'Demand': demand}
    for unit_index, unit in enumerate(Unit):
        info[f'Unit {unit} Power Produced'] = P[unit_index]
        info[f'Unit {unit} Cost'] = Fuel_Cost[unit_index]
    schedulling.append(info)

    # set baseline untuk ramp berikutnya
    baseline_prev = P.copy()

# ========= Ramp rates & generation limits (post-check) =========
ramp_rates = []
generation_limits = []
for i in range(1, len(all_outputs)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output  = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ========= Save to Excel =========
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-POZ) 6u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-POZ) 6u.xlsx'")

Load Counter: 1
Output for Power Demand 1263.00 MW
   Unit  Power Produced (MW)
0     1           499.999635
1     2           133.912234
2     3           283.283468
3     4            72.699259
4     5           198.105459
5     6            74.999945
Total Power Produced: 1263.00 MW
Total Fuel Cost: Rp 15343.149807989581
Computation Time: 0.03345298767089844 seconds
------------------------------------------------

All results saved to 'DOA (ED-POZ) 6u.xlsx'


In [4]:
import pandas as pd
import numpy as np
import random
import time
import math

# ========= Helpers: POZ parsing & feasibility ==========

def parse_poz_cell(cell):
    """
    Parse string seperti "[210 240], [350 380]" -> list of (210.0, 240.0), (350.0, 380.0)
    Jika kosong/NaN -> []
    """
    if isinstance(cell, str):
        pairs = []
        # ambil semua sub-string di dalam [..]
        for part in cell.replace(',', ' ').split(']'):
            if '[' in part:
                nums = part.split('[')[-1].strip().split()
                if len(nums) == 2:
                    try:
                        lo, hi = float(nums[0]), float(nums[1])
                        if lo > hi:
                            lo, hi = hi, lo
                        pairs.append((lo, hi))
                    except:
                        pass
        return pairs
    return []

def is_in_poz(val, poz_list):
    """True jika val berada di dalam salah satu interval terlarang."""
    for lo, hi in poz_list:
        if lo <= val <= hi:
            return True
    return False

def repair_out_of_poz(val, lb, ub, poz_list):
    """
    Jika val berada di interval terlarang, geser ke batas terdekat dari interval tersebut.
    Pastikan tetap di [lb, ub].
    """
    if not poz_list:
        return min(max(val, lb), ub)
    v = float(min(max(val, lb), ub))
    changed = True
    # perulangan singkat: kalau setelah digeser masih masuk interval lain
    for _ in range(5):
        moved = False
        for lo, hi in poz_list:
            if lo <= v <= hi:
                # pilih sisi terdekat
                to_lo = abs(v - lo)
                to_hi = abs(hi - v)
                v = lo - 1e-9 if to_lo <= to_hi else hi + 1e-9
                v = float(min(max(v, lb), ub))
                moved = True
        if not moved:
            break
    return v

def margin_up(val, ub, poz_list):
    """
    Headroom kenaikan tanpa memasuki POZ: jarak ke batas terdekat ke atas.
    """
    m = ub - val
    for lo, hi in poz_list:
        if val < lo:
            m = min(m, lo - val)
        elif lo <= val <= hi:
            m = min(m, max(0.0, hi - val))  # akan diperbaiki di tempat lain
        # jika val > hi: tidak mempengaruhi margin langsung
    return max(0.0, m)

def margin_down(val, lb, poz_list):
    """Headroom penurunan tanpa memasuki POZ."""
    m = val - lb
    for lo, hi in poz_list:
        if val > hi:
            m = min(m, val - hi)
        elif lo <= val <= hi:
            m = min(m, max(0.0, val - lo))
    return max(0.0, m)

def redistribute_to_meet_demand(P, demand, lb, ub, poz_lists, tol=1e-6):
    """
    Greedy ringan untuk menyesuaikan sum(P) -> demand tanpa masuk POZ.
    Menyebarkan selisih ke unit-unit berdasar margin naik/turun.
    """
    P = P.copy()
    diff = demand - np.sum(P)
    if abs(diff) <= tol:
        return P

    if diff > 0:  # butuh menaikkan
        # kapasitas kenaikan aman
        ups = np.array([margin_up(P[i], ub[i], poz_lists[i]) for i in range(len(P))])
        total_up = np.sum(ups)
        if total_up <= tol:
            return P  # tidak bisa, biar penalti objektif bekerja
        frac = np.minimum(1.0, np.maximum(0.0, ups / (total_up + 1e-12)))
        step = np.minimum(ups, diff * frac + 1e-12)
        P += step
    else:        # butuh menurunkan
        downs = np.array([margin_down(P[i], lb[i], poz_lists[i]) for i in range(len(P))])
        total_dn = np.sum(downs)
        if total_dn <= tol:
            return P
        frac = np.minimum(1.0, np.maximum(0.0, downs / (total_dn + 1e-12)))
        step = np.minimum(downs, (-diff) * frac + 1e-12)
        P -= step

    # rapikan dari POZ setelah redistribusi
    for i in range(len(P)):
        P[i] = repair_out_of_poz(P[i], lb[i], ub[i], poz_lists[i])
    return P

# ========= Load data ==========

data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/15-unit.xlsx',
    sheet_name='Table 3'
)

a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values.astype(float)
Maximum_Capacity = data['pmax'].values.astype(float)
ramp_up = data['rampup'].values.astype(float)
ramp_down = data['rampdown'].values.astype(float)
Unit = data['Unit'].values

# kolom opsional Pnow & poz
Pnow = data['Pnow'].values.astype(float) if 'Pnow' in data.columns else None
poz_lists = [parse_poz_cell(cell) for cell in (data['poz'].values if 'poz' in data.columns else [np.nan]*len(Unit))]

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd15u.xlsx')
power_demand = power_demand_data['load'].values

# ========= DOA dengan POZ ==========

def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    poz_lists,
                    pop=30, max_iter=100,
                    penalty_mismatch=1e4,
                    penalty_poz=1e6):
    dim = len(pmin)
    lb, ub = pmin, pmax

    def fobj(pos):
        # biaya kuadratik biasa
        fuel = np.sum(a + b * pos + c * pos**2)
        # penalti mismatch demand
        penalty = abs(demand - np.sum(pos)) * penalty_mismatch
        # penalti POZ (jarak minimum ke tepi interval jika berada di dalam)
        poz_pen = 0.0
        for i in range(dim):
            for lo, hi in poz_lists[i]:
                if lo <= pos[i] <= hi:
                    poz_pen += penalty_poz * min(pos[i]-lo, hi-pos[i] + 1e-12)
        return fuel + penalty + poz_pen

    # inisialisasi populasi (hindari POZ)
    x = np.random.uniform(lb, ub, (pop, dim))
    for p in range(pop):
        for i in range(dim):
            x[p, i] = repair_out_of_poz(x[p, i], lb[i], ub[i], poz_lists[i])

    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration 90% ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # batas + perbaikan POZ
                        x[j, h] = float(min(max(x[j, h], lb[h]), ub[h]))
                        x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]
                        x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])

            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation 10% ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                x[j, h] = float(min(max(x[j, h], lb[h]), ub[h]))
                x[j, h] = repair_out_of_poz(x[j, h], lb[h], ub[h], poz_lists[h])

        fbest_history[it] = fbest

    # ====== Final scaling & repairs ======
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    # clip + keluar dari POZ
    for i in range(dim):
        sbest[i] = float(min(max(sbest[i], lb[i]), ub[i]))
        sbest[i] = repair_out_of_poz(sbest[i], lb[i], ub[i], poz_lists[i])

    # Coba sesuaikan total agar match demand tanpa masuk POZ
    sbest = redistribute_to_meet_demand(sbest, demand, lb, ub, poz_lists)

    return sbest, max_iter

# ========= Fuel cost (tanpa VPE, tetap kuadratik) =========
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# ========= Main loop =========
all_outputs = []
schedulling = []
summary_rows = []

baseline_prev = Pnow if Pnow is not None else None

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    # jalankan optimisasi
    P, iterations = calculate_power(
        demand, a, b, c,
        Minimum_Capacity, Maximum_Capacity,
        ramp_up, ramp_down,
        poz_lists=poz_lists
    )

    # Terapkan ramp-rate relatif ke baseline:
    # - untuk titik pertama: dari Pnow (jika ada), kalau tidak dari P hasil (tidak diubah)
    # - selanjutnya: dari output sebelumnya
    if baseline_prev is not None:
        diff = P - baseline_prev
        up_mask, down_mask = diff > 0, diff < 0
        P[up_mask] = baseline_prev[up_mask] + np.minimum(diff[up_mask], ramp_up[up_mask])
        P[down_mask] = baseline_prev[down_mask] + np.maximum(diff[down_mask], -ramp_down[down_mask])
        # repair POZ setelah ramp
        for i in range(len(P)):
            P[i] = repair_out_of_poz(P[i], Minimum_Capacity[i], Maximum_Capacity[i], poz_lists[i])
        # coba paskan demand lagi
        P = redistribute_to_meet_demand(P, demand, Minimum_Capacity, Maximum_Capacity, poz_lists)

    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.2f} MW")
    print(f'Total Fuel Cost: Rp {total_Fuel_Cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': np.sum(P),
        'Total Fuel Cost': total_Fuel_Cost,
        'Computation Time': end_time - start_time
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': iterations
    })

    info = {'Demand': demand}
    for unit_index, unit in enumerate(Unit):
        info[f'Unit {unit} Power Produced'] = P[unit_index]
        info[f'Unit {unit} Cost'] = Fuel_Cost[unit_index]
    schedulling.append(info)

    # set baseline untuk ramp berikutnya
    baseline_prev = P.copy()

# ========= Ramp rates & generation limits (post-check) =========
ramp_rates = []
generation_limits = []
for i in range(1, len(all_outputs)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output  = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ========= Save to Excel =========
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-POZ) 15u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-POZ) 15u.xlsx'")

Load Counter: 1
Output for Power Demand 2630.00 MW
    Unit  Power Produced (MW)
0      1           453.637557
1      2           185.000000
2      3           103.859276
3      4            94.056783
4      5           176.044703
5      6           331.379979
6      7           428.790181
7      8           244.625848
8      9           148.603192
9     10           160.000000
10    11            68.712395
11    12            51.224616
12    13            83.277473
13    14            51.180535
14    15            49.607462
Total Power Produced: 2630.00 MW
Total Fuel Cost: Rp 33034.89799250317
Computation Time: 0.03575396537780762 seconds
------------------------------------------------

All results saved to 'DOA (ED-POZ) 15u.xlsx'
